In [1]:
from google.colab import files
uploaded = files.upload()

Saving training.1600000.processed.noemoticon.csv to training.1600000.processed.noemoticon.csv


In [2]:
import pandas as pd

df = pd.read_csv("your_file_name.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'your_file_name.csv'

In [ ]:
df = pd.read_csv("sentiment140.csv")
df.head()

In [3]:
import pandas as pd

df = pd.read_csv("your_uploaded_file_name.csv", encoding='latin-1')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'your_uploaded_file_name.csv'

In [4]:
import pandas as pd

df = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding='latin-1', header=None)
df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [5]:
df.columns = ['label', 'id', 'date', 'query', 'user', 'text']

# Keep only needed columns
df = df[['label', 'text']]

# Convert labels (4 → 1)
df['label'] = df['label'].replace(4,1)

df.head()

,label,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [6]:
df = df.sample(10000, random_state=42)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(stop_words='english', max_features=10000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [10]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

start = time.time()

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_bow, y_train)
y_pred_bow = lr.predict(X_test_bow)

end = time.time()

print("BoW + LR Accuracy:", accuracy_score(y_test, y_pred_bow))
print("BoW + LR F1:", f1_score(y_test, y_pred_bow))
print("Time:", end - start)

BoW + LR Accuracy: 0.7075
BoW + LR F1: 0.7196933397220892
Time: 1.890308141708374


In [11]:
start = time.time()

lr.fit(X_train_tfidf, y_train)
y_pred_tfidf_lr = lr.predict(X_test_tfidf)

end = time.time()

print("TF-IDF + LR Accuracy:", accuracy_score(y_test, y_pred_tfidf_lr))
print("TF-IDF + LR F1:", f1_score(y_test, y_pred_tfidf_lr))
print("Time:", end - start)

TF-IDF + LR Accuracy: 0.7115
TF-IDF + LR F1: 0.7237912876974629
Time: 0.31296730041503906


In [12]:
from sklearn.svm import LinearSVC

start = time.time()

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)

end = time.time()

print("TF-IDF + SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("TF-IDF + SVM F1:", f1_score(y_test, y_pred_svm))
print("Time:", end - start)

TF-IDF + SVM Accuracy: 0.7
TF-IDF + SVM F1: 0.7118155619596542
Time: 0.04867982864379883


In [13]:
sample = ["this product is amazing and works great"]

sample_vec = tfidf.transform(sample)
print("Prediction:", lr.predict(sample_vec))

Prediction: [1]
